# Inspect the fresh REDD audit
Published aggregates only; timestamp-count and threshold-duration ratios have different meanings.


In [1]:
from pathlib import Path
import json, csv
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/fresh/summary.json').exists())
data = root / 'data/fresh'
summary = json.loads((data / 'summary.json').read_text())
pooled = summary['pooled']
print('Original records:', f"{pooled['source_observations']:,}")
print('Timestamp union:', f"{pooled['union_timestamps']:,}")
print('Both-mains timestamps:', f"{pooled['both_mains_timestamps']:,}")

Original records: 56,341,629
Timestamp union: 7,755,931
Both-mains timestamps: 7,056,896


In [2]:
complete, mains, negative = (pooled[k] for k in ['all_channel_complete_timestamps', 'both_mains_timestamps', 'negative_residuals'])
assert abs(complete / mains - pooled['exact_complete_fraction_of_mains']) < 1e-14
assert abs(negative / complete - pooled['negative_residual_fraction']) < 1e-14
print(f'Exact complete / mains: {complete:,}/{mains:,} = {complete/mains:.4%}')
print(f'Negative / complete: {negative:,}/{complete:,} = {negative/complete:.4%}')
print('These are not sensor-loss rates or NILM accuracy scores.')

Exact complete / mains: 1,797,457/7,056,896 = 25.4709%
Negative / complete: 2,859/1,797,457 = 0.1591%
These are not sensor-loss rates or NILM accuracy scores.


In [3]:
for house in summary['houses']:
    connected, envelope = house['all_channel_connected_60_seconds'], house['mains_envelope_seconds']
    assert connected <= house['mains_connected_60_seconds'] <= envelope
    print(f"House {house['house']}: {connected:,}/{envelope:,} seconds = {connected/envelope:.2%}")
print('Threshold-connected duration is an availability proxy, not measured continuous uptime.')

House 1: 1,576,434/3,134,093 seconds = 50.30%
House 2: 1,201,449/3,026,449 seconds = 39.70%
House 3: 1,468,404/3,870,507 seconds = 37.94%
House 4: 1,633,642/4,145,366 seconds = 39.41%
House 5: 311,655/3,786,977 seconds = 8.23%
House 6: 1,112,552/2,022,745 seconds = 55.00%
Threshold-connected duration is an availability proxy, not measured continuous uptime.


In [4]:
sources = list(csv.DictReader((data / 'source_files.csv').open()))
assert len(sources) == 116
assert sum(int(r['source_rows']) for r in sources) == pooled['source_observations']
for key in ['malformed_rows', 'identical_to_first_duplicates', 'conflicting_with_first_duplicates']:
    assert sum(int(r[key]) for r in sources) == 0
assert sum(int(r['out_of_order_records']) for r in sources) == 47957
print('All 116 source-channel counts reconcile; malformed/duplicate/conflicting records = 0.')
print('Descending source adjacencies = 47,957; canonical ordering changes no measurement value.')

All 116 source-channel counts reconcile; malformed/duplicate/conflicting records = 0.
Descending source adjacencies = 47,957; canonical ordering changes no measurement value.
